## Operational Research — Process Inefficiency Analysis

**Purpose:** Identify and quantify process inefficiencies in the accounts payable workflow — specifically where the current manual system creates delays, errors, and financial risk.

**WM04 Part:** Part 6 — Operational Research

**WM04 Activity codes:** WA0125 (process analysis), WA0126 (bottleneck identification), WA0127 (recommendation formulation)

**Inputs:**
- `data/synthetic/sa_invoices.csv`
- `data/synthetic/sa_suppliers.csv`
- `data/synthetic/sa_bank_transactions.csv`

**Outputs:**
- `reports/ops_days_past_due.png`
- `reports/ops_payment_compliance.png`
- `reports/ops_bank_reconciliation.png`
- `reports/ops_vat_mismatch.png`
- Structured findings and recommendations in Section 5

**Author:** Ntsikelelo Nicholas Jantjie  
**Date:** June 2026

---

### Context

The finadmin360 project was born from a specific operational pain point: a South African finance administrator managing ~2 000 invoices per year using manual Excel-based workflows. This notebook moves beyond *describing* the data and into *diagnosing the process*: where exactly does the current workflow break down, how often, and what does it cost?

Operational research in a data science context means treating the business process as a system with inputs, throughput constraints, and measurable outputs. We identify:
1. Where invoices stay overdue longest (queue analysis)
2. Whether agreed payment terms are honoured (compliance analysis)
3. Which bank transactions cannot be matched to invoices (reconciliation gap)
4. The rate of VAT calculation errors (data quality in a regulatory context)

### Analytical Questions
1. Which supplier categories have the longest overdue periods?
2. How often do actual payment days exceed agreed payment terms?
3. What proportion of bank transactions are unreconciled?
4. Which suppliers have the highest VAT mismatch rate?

---
## 0. Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
BRAND_BLUE   = '#1f77b4'
BRAND_TEAL   = '#17becf'
BRAND_ORANGE = '#ff7f0e'
BRAND_RED    = '#d62728'
BRAND_GREEN  = '#2ca02c'

os.makedirs('reports', exist_ok=True)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Environment ready.')

---
## 1. Data Ingestion

In [ ]:
invoices = pd.read_csv(
    'data/synthetic/sa_invoices.csv',
    parse_dates=['invoice_date', 'due_date', 'payment_date']
)
suppliers = pd.read_csv('data/synthetic/sa_suppliers.csv')
bank      = pd.read_csv(
    'data/synthetic/sa_bank_transactions.csv',
    parse_dates=['transaction_date']
)

df = invoices.merge(suppliers, on='supplier_id', how='left')

print(f'Invoices:          {len(invoices):,}')
print(f'Suppliers:         {len(suppliers):,}')
print(f'Bank transactions: {len(bank):,}')
print(f'Merged df shape:   {df.shape}')

# Preview bank transaction schema
print('\nBank transaction columns:')
print(bank.dtypes)

---
## 2. Analysis 1 — Days Past Due by Category

**Operational question:** When invoices go overdue, how long do they stay overdue? A category that has a moderate late rate but very long overdue periods represents a worse cash-flow problem than a category with a higher late rate but quick resolution.

We use a box plot to show the distribution of overdue days — not just the mean — because a few extremely late invoices can mask a generally healthy picture.

In [ ]:
# ── Filter to overdue invoices only ──────────────────────────────────────────
overdue = df[df['days_past_due'] > 0].copy()

print(f'Total overdue invoices:      {len(overdue):,} ({len(overdue)/len(df):.1%} of all invoices)')
print(f'Median days past due:        {overdue["days_past_due"].median():.0f}')
print(f'90th percentile past due:    {overdue["days_past_due"].quantile(0.90):.0f}')
print(f'Max days past due:           {overdue["days_past_due"].max():.0f}')

In [ ]:
# ── Category medians for sort order ──────────────────────────────────────────
cat_median = (
    overdue.groupby('category_x')['days_past_due']
    .median()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(13, 6))

# Build colour map: red for median > 30 days, orange for > 15, blue otherwise
palette = {
    cat: BRAND_RED    if cat_median[cat] > 30
         else BRAND_ORANGE if cat_median[cat] > 15
         else BRAND_BLUE
    for cat in cat_median.index
}

sns.boxplot(
    data=overdue,
    x='category_x',
    y='days_past_due',
    order=cat_median.index,
    palette=palette,
    width=0.5,
    flierprops=dict(marker='o', markersize=3, alpha=0.4),
    ax=ax
)

# Reference line: 30-day threshold (commonly used for escalation)
ax.axhline(30, color='black', linestyle='--', linewidth=1.2,
           label='30-day escalation threshold')

ax.set_title('Days Past Due Distribution by Category (Overdue Invoices Only)',
             fontweight='bold')
ax.set_xlabel('Supplier Category')
ax.set_ylabel('Days Past Due')
ax.tick_params(axis='x', rotation=35)
ax.legend()

plt.tight_layout()
plt.savefig('reports/ops_days_past_due.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary table for the finding box ────────────────────────────────────────
overdue_summary = (
    overdue.groupby('category_x')['days_past_due']
    .agg(['median', 'mean', 'max', 'count'])
    .round(1)
    .sort_values('median', ascending=False)
    .rename(columns={'median': 'median_days', 'mean': 'mean_days',
                     'max': 'max_days', 'count': 'overdue_invoices'})
)
display(overdue_summary)

In [ ]:
# ── Generate finding text dynamically ────────────────────────────────────────
top_cat  = overdue_summary.index[0]
top_med  = overdue_summary.iloc[0]['median_days']
top_max  = overdue_summary.iloc[0]['max_days']

print(f'Top category: {top_cat} — median {top_med:.1f} days past due, max {top_max:.0f} days')

**Finding:** The highest-risk category has the longest median overdue period. Invoices in this category frequently exceed the 30-day escalation threshold, meaning the current manual follow-up process is not triggering soon enough.

**Recommendation:** Begin automated follow-up at day 10 past due for the highest-risk categories (not day 30). At next contract renewal, reduce payment terms by 15 days for suppliers in categories where median overdue exceeds 20 days. The finadmin360 platform's `ml_score_open_invoices` task already predicts which pending invoices are at risk — wire its output to an email alert triggered the morning of the due date.

---
## 3. Analysis 2 — Payment Terms Compliance

**Operational question:** When suppliers agree to Net-30 (or Net-60) payment terms, do they honour them? Non-compliance can be measured as the difference between the agreed terms and the actual days taken to pay.

This analysis covers only `PAID` invoices with a populated `payment_date`.

In [ ]:
# ── Paid invoices with payment dates ─────────────────────────────────────────
paid = df[(df['invoice_status'] == 'Paid') & df['payment_date'].notna()].copy()

paid['actual_days_to_pay'] = (paid['payment_date'] - paid['invoice_date']).dt.days
paid['days_delta']         = paid['actual_days_to_pay'] - paid['payment_terms_days_x']
paid['compliance_status']  = paid['days_delta'].apply(
    lambda d: 'Early' if d < -2 else ('On-time' if d <= 2 else 'Late')
)

print(f'Paid invoices with payment_date: {len(paid):,}')
print()
print('Compliance breakdown:')
print(paid['compliance_status'].value_counts())
print()
print(f'Median days delta:  {paid["days_delta"].median():.1f} days')
print(f'Mean days delta:    {paid["days_delta"].mean():.1f} days')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1 — Distribution of days delta
axes[0].hist(
    paid['days_delta'], bins=40,
    color=BRAND_BLUE, edgecolor='white', linewidth=0.3
)
axes[0].axvline(0, color=BRAND_RED, linestyle='--', linewidth=1.5,
                label='On-time threshold (±2 days)')
axes[0].axvline(paid['days_delta'].median(), color=BRAND_ORANGE,
                linestyle=':', linewidth=1.5,
                label=f'Median: {paid["days_delta"].median():.1f} days')
axes[0].set_title('Actual vs Agreed Payment Terms', fontweight='bold')
axes[0].set_xlabel('Days Delta (actual − agreed)\nNegative = early, Positive = late')
axes[0].set_ylabel('Invoice Count')
axes[0].legend()

# Panel 2 — Compliance by category
compliance_by_cat = (
    paid.groupby('category_x')['compliance_status']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=['Late', 'On-time', 'Early'], fill_value=0)
)
compliance_by_cat.plot(
    kind='barh',
    stacked=True,
    color=[BRAND_RED, BRAND_BLUE, BRAND_GREEN],
    edgecolor='white',
    ax=axes[1]
)
axes[1].set_title('Payment Compliance by Category', fontweight='bold')
axes[1].set_xlabel('Proportion of Paid Invoices')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('reports/ops_payment_compliance.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** A meaningful proportion of paid invoices are settled after the agreed payment terms. The days-delta distribution shows a right skew, meaning the tail of very-late payments is pulling the mean above the median.

**Recommendation:** Renegotiate payment terms with the two highest non-compliant categories. Specifically: reduce Net-60 terms to Net-45 at the next annual review, and introduce a 1.5% early payment discount to incentivise on-time settlement. The finadmin360 Gold layer (`gold_supplier_scorecard`) should be extended to track rolling 90-day compliance rate per supplier and surface this on the dashboard.

---
## 4. Analysis 3 — Bank Reconciliation Gap

**Operational question:** What proportion of bank transactions cannot be matched to a recorded invoice? Unreconciled transactions represent either (a) payments made outside the invoice system, (b) duplicate payments, or (c) fraud risk.

We use a fuzzy-amount match: a bank transaction is considered matched if there is a paid invoice from the same supplier with an amount within R10 of the bank transaction amount on the same day (±3 days tolerance).

In [ ]:
# ── Inspect bank transaction schema ──────────────────────────────────────────
print('Bank transaction schema:')
display(bank.head())
print(f'\nShape: {bank.shape}')
print(f'Date range: {bank["transaction_date"].min().date()} → {bank["transaction_date"].max().date()}')
print(f'Columns: {bank.columns.tolist()}')

In [ ]:
# ── Reconciliation matching logic ─────────────────────────────────────────────
#
# Strategy: for each bank transaction, look for a paid invoice where:
#   1. Amount matches within R50 tolerance (covers rounding and minor discounts)
#   2. Payment date is within 3 calendar days of transaction date
#
# This is a simplified reconciliation — production systems use invoice
# reference numbers. We demonstrate the gap-detection logic here.

paid_invoices = df[
    (df['invoice_status'] == 'Paid') & df['payment_date'].notna()
][['supplier_id', 'invoice_id', 'amount_incl_vat', 'payment_date']].copy()

# Determine which bank columns are available
print('Bank columns available for matching:')
print(bank.columns.tolist())

# Adapt column names to what actually exists in the data
# Common column names to check
amount_col = next((c for c in ['amount', 'transaction_amount', 'debit_amount']
                   if c in bank.columns), None)
ref_col    = next((c for c in ['reference', 'transaction_reference', 'description']
                   if c in bank.columns), None)

print(f'\nUsing amount column: {amount_col}')
print(f'Using reference column: {ref_col}')

In [ ]:
# ── Amount-based reconciliation (if amount column exists) ─────────────────────
if amount_col:
    bank_debits = bank[bank[amount_col] < 0].copy() if (bank[amount_col] < 0).any() \
                  else bank.copy()
    bank_debits['abs_amount'] = bank_debits[amount_col].abs()

    matched_ids = set()
    for _, txn in bank_debits.iterrows():
        window_start = txn['transaction_date'] - pd.Timedelta(days=3)
        window_end   = txn['transaction_date'] + pd.Timedelta(days=3)
        candidates = paid_invoices[
            (paid_invoices['payment_date'] >= window_start) &
            (paid_invoices['payment_date'] <= window_end) &
            (abs(paid_invoices['amount_incl_vat'] - txn['abs_amount']) <= 50)
        ]
        if not candidates.empty:
            matched_ids.add(txn.name)

    total_txns     = len(bank_debits)
    matched_txns   = len(matched_ids)
    unmatched_txns = total_txns - matched_txns
    unmatched_pct  = unmatched_txns / total_txns if total_txns > 0 else 0

    print(f'Total bank transactions (debit):  {total_txns:,}')
    print(f'Matched to invoices:              {matched_txns:,} ({matched_txns/total_txns:.1%})')
    print(f'Unreconciled:                     {unmatched_txns:,} ({unmatched_pct:.1%})')
else:
    print('Amount column not found. Please inspect bank.columns and adjust amount_col.')
    print('Skipping reconciliation calculation.')
    unmatched_pct = None

In [ ]:
# ── Monthly unreconciled transaction trend ────────────────────────────────────
if amount_col:
    bank_debits['month'] = bank_debits['transaction_date'].dt.to_period('M')
    bank_debits['matched'] = bank_debits.index.isin(matched_ids)

    monthly_recon = (
        bank_debits.groupby('month')['matched']
        .agg(['sum', 'count'])
        .rename(columns={'sum': 'matched', 'count': 'total'})
    )
    monthly_recon['unmatched']     = monthly_recon['total'] - monthly_recon['matched']
    monthly_recon['unmatched_pct'] = monthly_recon['unmatched'] / monthly_recon['total']
    monthly_recon['month_str']     = monthly_recon.index.astype(str)

    fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

    axes[0].bar(monthly_recon['month_str'], monthly_recon['matched'],
                label='Matched', color=BRAND_GREEN, alpha=0.85, edgecolor='white')
    axes[0].bar(monthly_recon['month_str'], monthly_recon['unmatched'],
                bottom=monthly_recon['matched'],
                label='Unreconciled', color=BRAND_RED, alpha=0.85, edgecolor='white')
    axes[0].set_title('Monthly Bank Transaction Reconciliation Status', fontweight='bold')
    axes[0].set_ylabel('Transaction Count')
    axes[0].legend()

    axes[1].plot(monthly_recon['month_str'], monthly_recon['unmatched_pct'],
                 color=BRAND_RED, linewidth=2, marker='o', markersize=5)
    axes[1].axhline(0.05, color='black', linestyle='--', linewidth=1,
                    label='5% acceptable threshold')
    axes[1].set_title('Unreconciled Transaction Rate by Month', fontweight='bold')
    axes[1].set_ylabel('Unreconciled Rate')
    axes[1].set_xlabel('Month')
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    axes[1].legend()

    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('reports/ops_bank_reconciliation.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping reconciliation chart — amount column not found.')

**Finding:** A proportion of bank transactions cannot be matched to recorded invoices using amount and date proximity. This reconciliation gap represents transactions that either were not captured in the invoice system at all, or were captured with incorrect amounts (data entry errors). Any unreconciled rate above 5% in a given month is a material control weakness.

**Recommendation:** Implement mandatory invoice reference numbers on all bank payment descriptions — this enables exact-match reconciliation and eliminates the need for fuzzy matching entirely. In the interim, the finadmin360 pipeline should flag any bank debit > R1 000 that remains unreconciled for more than 5 business days and route it to a daily exception report for the administrator.

---
## 5. Analysis 4 — VAT Mismatch Rate by Supplier

**Operational question:** South African VAT is 15% of the excl. amount. Invoices where VAT ≠ 15% × excl. amount (within a R0.02 rounding tolerance) indicate either supplier billing errors or data entry mistakes. Both create SARS audit risk.

The Silver layer in dbt already flags these as `VAT_MISMATCH`. Here we quantify the problem at the supplier level.

In [ ]:
# ── Compute VAT validation in Python (mirrors dbt Silver logic) ───────────────
df['vat_expected']   = (df['amount_excl_vat'] * 0.15).round(2)
df['vat_delta']      = (df['vat_amount'] - df['vat_expected']).abs()
df['vat_valid']      = df['vat_delta'] < 0.02
df['vat_mismatch']   = ~df['vat_valid']

overall_mismatch_rate = df['vat_mismatch'].mean()
print(f'Overall VAT mismatch rate: {overall_mismatch_rate:.2%}')
print(f'Total mismatch invoices:   {df["vat_mismatch"].sum():,}')

In [ ]:
# ── Mismatch by supplier ──────────────────────────────────────────────────────
vat_by_supplier = (
    df.groupby(['supplier_id', 'supplier_name'])['vat_mismatch']
    .agg(['mean', 'sum', 'count'])
    .rename(columns={'mean': 'mismatch_rate', 'sum': 'mismatch_count', 'count': 'total_invoices'})
    .reset_index()
    .sort_values('mismatch_rate', ascending=False)
)

# Top 15 worst suppliers
top_vat = vat_by_supplier[vat_by_supplier['total_invoices'] >= 5].head(15)

print('Top 15 suppliers by VAT mismatch rate (min 5 invoices):')
display(top_vat.style.format({
    'mismatch_rate': '{:.1%}',
    'mismatch_count': '{:.0f}',
    'total_invoices': '{:.0f}'
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1 — Top suppliers by mismatch rate
top10 = top_vat.head(10)
colors = [BRAND_RED if r > 0.5 else BRAND_ORANGE if r > 0.2 else BRAND_BLUE
          for r in top10['mismatch_rate']]

axes[0].barh(top10['supplier_name'], top10['mismatch_rate'],
             color=colors, edgecolor='white')
axes[0].set_title('Top 10 Suppliers by VAT Mismatch Rate', fontweight='bold')
axes[0].set_xlabel('VAT Mismatch Rate')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].invert_yaxis()

# Panel 2 — Mismatch rate by category
vat_by_cat = (
    df.groupby('category_x')['vat_mismatch']
    .mean()
    .sort_values(ascending=False)
)
colors2 = [BRAND_RED if r > 0.3 else BRAND_ORANGE if r > 0.1 else BRAND_BLUE
           for r in vat_by_cat]

axes[1].bar(vat_by_cat.index, vat_by_cat.values,
            color=colors2, edgecolor='white')
axes[1].axhline(overall_mismatch_rate, color='black', linestyle='--',
                linewidth=1.2, label=f'Overall: {overall_mismatch_rate:.1%}')
axes[1].set_title('VAT Mismatch Rate by Category', fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Mismatch Rate')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend()

plt.suptitle('VAT Calculation Quality Analysis — finadmin360', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('reports/ops_vat_mismatch.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** VAT mismatch is concentrated in a subset of suppliers and categories. A systematic mismatch rate above 10% for any single supplier likely indicates a billing template error on the supplier's side rather than isolated data entry mistakes.

**Recommendation:** Raise a formal billing accuracy query with any supplier whose rolling 90-day VAT mismatch rate exceeds 15%. These invoices cannot be submitted for VAT input tax claims until corrected, creating a direct tax liability. The dbt Silver layer already flags these rows as `VAT_MISMATCH` — the Gold layer should aggregate this into the `gold_vat_summary` model and surface the count on the dashboard.

---
## 6. Consolidated Operational Findings & Recommendations

In [ ]:
print('=' * 72)
print('FINADMIN360 — OPERATIONAL RESEARCH FINDINGS & RECOMMENDATIONS')
print('=' * 72)

top_overdue_cat = overdue_summary.index[0]
top_overdue_med = overdue_summary.iloc[0]['median_days']

print(f'''
FINDING 1 — Overdue queue depth exceeds 30 days in highest-risk categories
  Category "{top_overdue_cat}" has a median overdue period of
  {top_overdue_med:.0f} days. Under the current manual workflow, follow-up
  is not initiated until well past this point.

  RECOMMENDATION: Configure the daily pipeline to trigger an email alert
  when days_past_due > 10 for any invoice in a high-risk category.
  At next contract renewal, reduce payment terms by 15 days for
  the two worst-performing categories.

FINDING 2 — Payment terms non-compliance creates downstream cash-flow uncertainty
  A material share of paid invoices settle after the agreed terms.
  This makes 13-week cash-flow forecasting unreliable, because
  the administrator cannot rely on payment dates as committed.

  RECOMMENDATION: Introduce a 1.5% early-payment discount for on-time
  settlement. Track rolling 90-day compliance rate per supplier in the
  gold_supplier_scorecard dbt model and include it in quarterly
  supplier reviews.

FINDING 3 — Bank reconciliation gap requires structured invoice referencing
  Fuzzy amount-based matching leaves a reconciliation gap because
  payments are not linked to invoice reference numbers in bank
  transaction descriptions.

  RECOMMENDATION: Mandate invoice reference numbers in all bank payment
  descriptions. Add an "unreconciled > 5 business days" exception report
  to the daily Airflow pipeline output.

FINDING 4 — VAT mismatch rate creates SARS audit exposure
  A concentrated subset of suppliers regularly submit invoices with
  incorrect VAT calculations. These invoices cannot be used for
  input tax claims without correction.

  RECOMMENDATION: Send a billing accuracy notice to suppliers with
  >15% VAT mismatch over 90 days. Extend the gold_vat_summary dbt
  model to flag these suppliers automatically and surface the count
  on the finadmin360 dashboard.
''')
print('=' * 72)

---
## 7. Export

```bash
jupyter nbconvert --to pdf notebooks/05_operational_research.ipynb
mv notebooks/05_operational_research.pdf reports/
```

---
*End of 05_operational_research.ipynb*